# 06: Full Model

`AlphaFold2FromScratch` connects the components from the previous notebooks. It accepts a prepared feature batch and returns residue frames, a C-alpha backbone trace, distance logits, and confidence logits.

The model class is intentionally small. Its job is orchestration rather than implementing another neural-network block.

**Read alongside:** `../src/af2_from_scratch/model.py`

## Stage map

```text
prepared features from feature_extraction.py
                    |
                    v
              InputEmbedder
                m, z, e
                    |
       +------------v-------------------+
       | RecyclingEmbedder + Evoformer  |
       | start again from initial m, z  |
       | and inject previous outputs    |
       +------------+-------------------+
                    |
             query row m[0]
                    |
                    v
             single features s
                    |
                    v
             StructureModule
                    |
          residue frames + C-alpha trace

final z --------------------> distogram logits
refined s ------------------> pLDDT logits
```

In [ ]:
import sys

import torch

sys.path.insert(0, "../src")
torch.manual_seed(0)

from af2_from_scratch import AF2Config, AlphaFold2FromScratch
from af2_from_scratch.feature_extraction import msa_features, sample_batch

## 1. Build a small model-ready batch

Feature extraction happens outside the model. The model receives tensors instead of raw A3M text, which keeps parsing and neural-network computation separate.

This notebook uses reduced widths and depths so the complete forward pass runs quickly.

In [ ]:
cfg = AF2Config(
    c_m=32,
    c_z=32,
    c_e=16,
    c_s=64,
    heads=4,
    pair_heads=2,
    ipa_heads=2,
    c_hidden=8,
    n_evo=1,
    n_extra=1,
    n_ipa=1,
    n_clu=16,
    n_ext=16,
    recycles=1,
)
features = msa_features("../examples/tautomerase/alignment.a3m")
batch = sample_batch(
    features,
    cfg.n_clu,
    cfg.n_ext,
    mask_p=0.0,
    seed=0,
)
for name, value in batch.items():
    print(f"{name:16s} {tuple(value.shape)}")

## 2. Refine representations through recycling

`recycles=1` means two Evoformer passes: one initial pass and one recycled pass.

Every pass starts from the original embedded `m` and `z`. The recycling embedder then adds normalized outputs from the preceding pass. Starting from the original embeddings avoids accidentally accumulating the entire previous representation twice.

The recycled tensors are detached, so gradients do not cross recycle boundaries. The final pass still receives gradients normally.

In [ ]:
model = AlphaFold2FromScratch(cfg).eval()
with torch.no_grad():
    output = model(batch)

print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
for name, value in output.items():
    print(f"{name:16s} {tuple(value.shape)}")

## 3. Read the outputs

```text
T              (R, 4, 4)    one rigid frame per residue
ca             (R, 3)       C-alpha frame origins

disto_logits   (R, R, 64)   unnormalized scores for distance bins
plddt_logits   (R, 50)      unnormalized confidence-bin scores
```

The distance head receives a symmetrized pair representation, so residue pair `(i, j)` has the same logits as `(j, i)`. Applying `softmax` converts either set of logits into probabilities.

The `ca` output is a simplified backbone trace, not an all-atom structure.

**Recap:** The full model embeds features, repeats Evoformer refinement with recycling, converts the query MSA row into single features, predicts residue frames, and applies distance and confidence heads.

Notebook 07 introduces the training objectives and a single-protein distillation experiment.